# Fig. 7f: information capacity of the ground truth under the undersampling and the selection bias of each spike sorter (evoked model)

**author**: laquitainesteeve@gmail.com

**purpose**: reproduce panel f of the paper: manifold information capacity, divided by its lower bound, of (red) random samples of the ground truth neurons with as many neurons as each sorter has sorted single-units (undersampling) and (white) samples that also match the distribution of the unit types of the sorter (undersampling and selection bias). Source data and svg are saved in figures/8_source_data/fig7f/.

**method (as in the paper's notebook fig7_F_capacity_vs_sampling_revised.ipynb)**
* the capacity of each population is computed on 200 bootstraps of the sampled neurons (population responses to the 360 stimulus directions, 200 latents, `neural_manifolds_replicaMFT`); chance is 1
* the bootstrapped capacities are read from the csv file of the paper (`igeometrics_by_sampling.csv`); they are **not recomputed** here (the capacity analysis needs the `mftma` package)
* box plots (notch, 200 bootstraps); red dashed line: mean capacity of the whole ground truth; blue dashed line: chance (1)
* comparison with the whole ground truth: two-sample t-test between the 200 bootstrapped capacities of the sample and those of the ground truth (as in the paper's notebook)

In [1]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import pandas as pd
import spikeinterface as si
from matplotlib import pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# set project path
PROJ_PATH = "/home/steeve/steeve/epfl/code/spikebias/"
os.chdir(PROJ_PATH)

# custom package
import seaborn as sns
from scipy.stats import ttest_ind

# paths (bootstrapped manifold capacities of the ground truth and of the sorted populations, evoked simulation, 1 hour)
NEURAL_CODE_PATH = "dataset/01_intermediate/neural_code/npx_evoked/"
IGEOM_SAMP = NEURAL_CODE_PATH + "igeometrics_by_sampling.csv"

# FIGURE SETTINGS
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 6  # 5-7 with Nature neuroscience as reference
plt.rcParams["lines.linewidth"] = 0.5
plt.rcParams["axes.linewidth"] = 0.5
plt.rcParams["axes.spines.top"] = False
plt.rcParams["xtick.major.width"] = 0.5
plt.rcParams["xtick.minor.width"] = 0.5
plt.rcParams["ytick.major.width"] = 0.5
plt.rcParams["ytick.minor.width"] = 0.5
plt.rcParams["xtick.major.size"] = 3.5 * 1.1
plt.rcParams["xtick.minor.size"] = 2 * 1.1
plt.rcParams["ytick.major.size"] = 3.5 * 1.1
plt.rcParams["ytick.minor.size"] = 2 * 1.1
legend_cfg = {"frameon": False, "handletextpad": 0.5}

# save config
savefig_cfg = {"transparent": True, "dpi": 300}

In [ ]:
# Nature figure guidelines for every saved figure: Arial, 5-7 pt, editable text, images of at least 300 dpi, a .pdf next to each .svg
import os, sys
sys.path.insert(0, os.getcwd())     # the repository (the notebook moved there above)
from src.nodes import figure_style
figure_style.apply()

## Load the bootstrapped capacities

In [2]:
%%time

igeom_df = pd.read_csv(IGEOM_SAMP)
SORTERS = ["KS4", "KS3", "KS2.5", "KS2", "KS", "HS"]
df = igeom_df[["Sorter", "Sampling scheme", "Capacity"]].copy()
df["sampling_scheme"] = df["Sampling scheme"].map({"random": "undersampling", "biased": "selection bias"})
df.loc[df["Sorter"] == "ground_truth", "sampling_scheme"] = "none (whole ground truth)"
df["population"] = df["Sorter"].replace({"ground_truth": "ground truth"})
df["bootstrap"] = df.groupby(["Sorter", "Sampling scheme"]).cumcount()
df = df.rename(columns={"Capacity": "capacity_over_lower_bound"})[["population", "sampling_scheme", "bootstrap", "capacity_over_lower_bound"]]
print(df.groupby(["population", "sampling_scheme"], sort=False)["capacity_over_lower_bound"].agg(["count", "median"]).round(3))

                                        count  median
population   sampling_scheme                         
ground truth none (whole ground truth)    200   1.249
KS4          undersampling                200   1.251
             selection bias               200   1.173
KS3          undersampling                200   1.251
             selection bias               200   1.203
KS2.5        undersampling                200   1.254
             selection bias               200   1.206
KS2          undersampling                200   1.260
             selection bias               200   1.173
KS           undersampling                200   1.253
             selection bias               200   1.217
HS           undersampling                200   1.253
             selection bias               200   1.209
CPU times: user 6.43 ms, sys: 0 ns, total: 6.43 ms
Wall time: 6.44 ms


## Capacity of the samples against the whole ground truth

In [3]:
gt = df[df["population"] == "ground truth"]["capacity_over_lower_bound"]
rows = []
for sorter in SORTERS:
    for scheme in ["undersampling", "selection bias"]:
        c = df[(df["population"] == sorter) & (df["sampling_scheme"] == scheme)]["capacity_over_lower_bound"]
        t, p = ttest_ind(c, gt)
        rows.append(dict(population=sorter, sampling_scheme=scheme, median_capacity=c.median(), change_of_capacity_above_lower_bound_percent=100 * (c.median() - gt.median()) / (gt.median() - 1), t_statistic=t, p_value=p))
stats = pd.DataFrame(rows)
display(stats.round(4))
for scheme in ["undersampling", "selection bias"]:
    s = stats[stats["sampling_scheme"] == scheme]
    print(f"{scheme}: average change of the capacity above the lower bound {s['change_of_capacity_above_lower_bound_percent'].mean():.1f}% | t from {s['t_statistic'].min():.2f} to {s['t_statistic'].max():.2f} | p from {s['p_value'].min():.1e} to {s['p_value'].max():.1e}")

,population,sampling_scheme,median_capacity,change_of_capacity_above_lower_bound_percent,t_statistic,p_value
0,KS4,undersampling,1.2512,1.0152,-0.1302,0.8965
1,KS4,selection bias,1.1733,-30.3259,-16.1440,0.0000
2,KS3,undersampling,1.2511,0.9432,-0.0253,0.9798
3,KS3,selection bias,1.2028,-18.4774,-10.0761,0.0000
4,KS2.5,undersampling,1.2542,2.1942,0.5819,0.5610
5,KS2.5,selection bias,1.2056,-17.3319,-8.6155,0.0000
6,KS2,undersampling,1.2596,4.3818,0.7151,0.4750
7,KS2,selection bias,1.1726,-30.5946,-17.7016,0.0000
8,KS,undersampling,1.2526,1.5781,0.6553,0.5126
9,KS,selection bias,1.2165,-12.9408,-9.6586,0.0000


undersampling: average change of the capacity above the lower bound 2.0% | t from -0.13 to 0.72 | p from 4.7e-01 to 9.8e-01
selection bias: average change of the capacity above the lower bound -20.9% | t from -17.70 to -7.39 | p from 3.9e-52 to 8.5e-13


## Plot, same as the paper's panel f

In [4]:
# paper geometry (page 22 of the paper): canvas 116.8 x 120.5 pt, Arial 5 pt text, editable (src/nodes/validation/capacity_plots.py)
from src.nodes.validation.capacity_plots import plot_sampling_capacity

def plot_source_data(df: pd.DataFrame):
    return plot_sampling_capacity(df)

## Plot and save the figure and the source data

In [5]:
%%time

# plot
fig = plot_source_data(df)

# save figure
os.makedirs("figures/8_source_data/fig7f", exist_ok=True)
plt.savefig("figures/8_source_data/fig7f/fig7f.svg", transparent=True)   # the .pdf is written next to the .svg

# save source data (sheet 1: capacity of the 200 bootstraps of each population and sampling scheme; sheet 2: comparison with the whole ground truth)
with pd.ExcelWriter("figures/8_source_data/fig7f/fig7f.xlsx") as writer:
    df.to_excel(writer, sheet_name="capacity_bootstraps", index=False)
    stats.to_excel(writer, sheet_name="vs_ground_truth", index=False)

## Check: replot from the saved source data

In [6]:
%%time

df = pd.read_excel("figures/8_source_data/fig7f/fig7f.xlsx", sheet_name="capacity_bootstraps")
fig = plot_source_data(df)